# Spaces Pipeline Pro — GRPO Training

End-to-end RL training of an LLM agent on the Spaces Pipeline environment.

**Stack:** Unsloth + TRL GRPO + Qwen 2.5 1.5B-Instruct + LoRA

**Curriculum:** 4 phases (warmup → discovery → drift → adversarial)

**Hardware:** Free Colab T4 viable for 1.5B; A100 recommended for 3B+

## 1. Install dependencies

In [ ]:
!pip install -q unsloth trl transformers datasets torch
!pip install -q openenv-core[core] huggingface_hub gradio_client

## 2. Clone the env repo

In [ ]:
!git clone https://github.com/rishabh16196/spaces-pipeline-env.git
%cd spaces-pipeline-env
!pip install -e .

## 3. Start the env server (in background)

In [ ]:
import subprocess
import time

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(3)
print('Env server started on :8000')

## 4. Quick env smoke test

In [ ]:
import asyncio
from spaces_pipeline_env import SpacesPipelineEnv, SpacesPipelineAction

async def smoke():
    env = SpacesPipelineEnv(base_url='http://localhost:8000')
    await env.connect()
    r = await env.reset(task='audio_summarize_hindi_001', seed=42)
    print('Task:', r.observation.task_id)
    print('Description:', r.observation.task_description[:80])
    a = SpacesPipelineAction(action_type='search_spaces', payload={'query': 'whisper', 'top_k': 3})
    r = await env.step(a)
    print('Search results:', len(r.observation.last_search_results))
    await env.close()

await smoke()

## 5. Phase 1: Warmup (5K rollouts on easy tasks, dense rewards)

In [ ]:
!python scripts/train_grpo.py --phase 1 --steps 5000 --output-dir outputs/phase1

## 6. Phase 2: Discovery (medium tasks, persona rotation)

In [ ]:
!python scripts/train_grpo.py --phase 2 --steps 5000 --output-dir outputs/phase2

## 7. Phase 3: Drift (introduce schema changes mid-episode)

In [ ]:
!python scripts/train_grpo.py --phase 3 --steps 10000 --output-dir outputs/phase3

## 8. Phase 4: Adversarial (sparse rewards, full curriculum)

In [ ]:
!python scripts/train_grpo.py --phase 4 --steps 10000 --output-dir outputs/phase4

## 9. Held-out evaluation

In [ ]:
!python scripts/evaluate.py --agent trained --model-path outputs/phase4 --output eval_results.json

## 10. Plot reward curves

In [ ]:
import json
import matplotlib.pyplot as plt

results = json.load(open('eval_results.json'))
tasks = [r['task_id'] for r in results['results']]
grades = [r['grade'] for r in results['results']]

plt.figure(figsize=(10, 5))
plt.bar(tasks, grades)
plt.axhline(y=0.5, color='r', linestyle='--', label='Pass threshold')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Grade')
plt.title(f'Held-out evaluation (avg={results["avg_grade"]:.3f}, pass_rate={results["pass_rate"]:.1%})')
plt.legend()
plt.tight_layout()
plt.savefig('eval_results.png')
plt.show()